# Simulació de Propagació d'Incendi Forestal — Sessió 2

## Descripció

En aquesta sessió modelitzem la propagació d'un incendi forestal amb un **autòmat cel·lular m:n-CA^k** representable amb SDL.  
Treballem amb **dues capes principals** carregades des de fitxers en format IDRISI32 i IDRISI31 vectorial, i una tercera capa que registra la propagació dinàmica.

### Objectiu
Simular com s'estén un incendi a través d'un terreny heterogeni, considerant la humitat i la vegetació com a variables principals, amb la possibilitat d'incorporar l'efecte del vent.

---

### Assumpcions extretes de l'enunciat

1. La **vegetació** representa les **hores de combustió** d'una cel·la (ex: 10 → la cel·la triga 10 passos a cremar-se completament).
2. La **humitat** representa les **hores que han de passar** des que una cel·la adjacent crema fins que el foc s'inicia a la cel·la (eliminació de la humitat prèvia a la combustió).
3. Un cop eliminada la humitat, la vegetació comença a cremar i la cel·la **ja pot propagar el foc** als seus veïns.
4. El model ha de tenir **com a mínim dues capes que interactuïn** (vegetació i humitat) i una **tercera capa de propagació** amb tres estats: cremat, en procés de cremar-se i pendent de cremar-se.

### Assumpcions pròpies de disseny

5. La propagació segueix un **veïnatge de Moore (8 veïns)**: s'ha triat perquè permet la propagació en diagonal, més realista que un veïnatge de Von Neumann.
6. La probabilitat base de propagació és del **85%** per pas i veí: introdueix estocàsticitat per reflectir la variabilitat real dels incendis.
7. Les zones amb humitat ≥ 5.0 es tracten com a **masses d'aigua** (no cremen mai): s'usa el valor màxim de la capa d'humitat com a indicador de terreny no inflamable.
8. La graella és de **100×100 cel·les** generada proceduralment quan no hi ha fitxers externs disponibles.
9. Es poden definir **múltiples focus d'ignició** simultanis per explorar escenaris de diversos orígens.

---
## 1. Importacions i constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import os
import math

# ── Estats de les cel·les ──────────────────────────────────────
class CellState:
    EMPTY   = 0   # pendent de cremar (vegetació sana)
    MOIST   = 1   # escalfant-se (eliminant humitat)
    BURNING = 2   # cremant activament → propaga foc
    BURNED  = 3   # ja cremat (cendra)
    WATER   = 4   # massa d'aigua (no crema mai)

# ── Mapa de codis de terreny (Initialize.img) → humitat base (hores) ──
TERRAIN_HUMIDITY = {
    "LL": 0,   # llis / sense vegetació
    "CT": 1,
    "TE": 1,
    "GR": 2,   # gespa / herba
    "CC": 2,
    "AA": 3,
    "BS": 3,   # bosc sec
    "BN": 4,   # bosc normal
    "BC": 5,   # bosc costaner (més humit)
    "CAT": 2,  # Catalunya genèric
}

# ── Mapa de polígon ID (vegetation.vec) → hores de combustió ──
POLYGON_VEGETATION = {
    1: 10,    # polígon 1 → vegetació densa
    2: 5,     # polígon 2 → vegetació mitjana
    20: 3,    # polígon 20 → vegetació baixa
}
DEFAULT_VEGETATION = 4  # valor per defecte

# ── Colors i llegenda ──────────────────────────────────────────
CMAP_COLORS = ["#639922", "#F4C0D1", "#EF9F27", "#444441", "#40a4df"]
CMAP = mcolors.ListedColormap(CMAP_COLORS)
NORM = mcolors.BoundaryNorm([0, 1, 2, 3, 4, 5], CMAP.N)
LEGEND_PATCHES = [
    mpatches.Patch(color=CMAP_COLORS[0], label="Vegetació (sana)"),
    mpatches.Patch(color=CMAP_COLORS[1], label="Escalfant-se (humida)"),
    mpatches.Patch(color=CMAP_COLORS[2], label="En flames"),
    mpatches.Patch(color=CMAP_COLORS[3], label="Cremada"),
    mpatches.Patch(color=CMAP_COLORS[4], label="Massa d'aigua"),
]

print("Importacions completades.")

---
## 2. Mode d'operació — Origen de les dades

Tria aquí si vols usar els **fitxers reals** (IDRISI32/31 adjunts) o generar dades **proceduralment** (útil per provar el model sense fitxers externs).

In [ ]:
# ══════════════════════════════════════════════════════
#  CONFIGURA AQUÍ EL MODE D'OPERACIÓ
# ══════════════════════════════════════════════════════
MODE = "fitxers"   # "fitxers"    → usa Initialize.img + vegetation.vec
                   # "procedural" → genera dades sintètiques 100×100

# ── Paràmetres mode procedural (només si MODE = "procedural") ──
ROWS_PROC  = 100
COLS_PROC  = 100
NUM_LLACS  = 3
RADIO_LLAC = 10
SEMILLA    = 42
# ══════════════════════════════════════════════════════

BASE_DIR = "dades"

if MODE == "procedural":
    import math
    np.random.seed(SEMILLA)

    # Vegetació: gradient radial (més densa al centre) + soroll
    x = np.linspace(0, 1, COLS_PROC)
    y = np.linspace(0, 1, ROWS_PROC)
    X, Y = np.meshgrid(x, y)
    dist_centre = np.sqrt((X - 0.5)**2 + (Y - 0.5)**2)
    veg_proc = 5 + 10 * (1 - dist_centre) + np.random.normal(0, 2, (ROWS_PROC, COLS_PROC))
    veg_proc = np.clip(veg_proc, 1, 20).astype(np.float32)

    # Humitat: base baixa + llacs circulars (humitat = 5.0 → no crema)
    hum_proc = np.random.uniform(0, 2, (ROWS_PROC, COLS_PROC)).astype(np.float32)
    for _ in range(NUM_LLACS):
        cx = np.random.randint(RADIO_LLAC, ROWS_PROC - RADIO_LLAC)
        cy = np.random.randint(RADIO_LLAC, COLS_PROC - RADIO_LLAC)
        for i in range(max(0, cx - RADIO_LLAC), min(ROWS_PROC, cx + RADIO_LLAC)):
            for j in range(max(0, cy - RADIO_LLAC), min(COLS_PROC, cy + RADIO_LLAC)):
                dist = math.sqrt((i - cx)**2 + (j - cy)**2)
                if dist <= RADIO_LLAC * 0.8:
                    hum_proc[i, j] = 5.0
                elif dist <= RADIO_LLAC:
                    hum_proc[i, j] = max(hum_proc[i, j], 4.5)
    hum_proc = np.clip(hum_proc, 0, 5)

    print(f"✅ Mode PROCEDURAL — graella {ROWS_PROC}×{COLS_PROC}, {NUM_LLACS} llac(s)")
    print(f"   Vegetació  min/max: {veg_proc.min():.1f} / {veg_proc.max():.1f} hores")
    print(f"   Humitat    min/max: {hum_proc.min():.1f} / {hum_proc.max():.1f} hores")
else:
    veg_proc = None
    hum_proc = None
    print(f"✅ Mode FITXERS — es llegiran Initialize.img i vegetation.vec")

---
## 3. Lectura de fitxers IDRISI32 — Capa d'Humitat

Els fitxers `Initialize.doc` (capçalera) i `Initialize.img` (dades) segueixen el format IDRISI32.  
El codi suporta tant el format **ASCII** (fitxers de pràctica) com el format **binari** (fitxers generats proceduralment).  
Si els fitxers originals no existeixen, s'utilitzen les capes generades a la secció anterior.

In [ ]:
def parse_idrisi_doc(path):
    """Llegeix una capçalera IDRISI32 (.doc) i retorna un dict clau:valor."""
    meta = {}
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip().lower()] = val.strip()
    return meta


def parse_idrisi_img_ascii(path):
    """Llegeix un fitxer de dades IDRISI32 ASCII (.img). Retorna llista de strings."""
    values = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            v = line.strip()
            if v:
                values.append(v)
    return values


def build_humidity_grid_ascii(img_values, rows, cols):
    """Construeix la quadrícula de humitat des de codis ASCII, repetint cíclicament."""
    grid = np.zeros((rows, cols), dtype=float)
    n = len(img_values)
    for r in range(rows):
        for c in range(cols):
            code = img_values[(r * cols + c) % n].upper()
            grid[r, c] = TERRAIN_HUMIDITY.get(code, 1)
    return grid


if MODE == "fitxers":
    DOC_PATH = os.path.join(BASE_DIR, "Initialize.doc")
    IMG_PATH = os.path.join(BASE_DIR, "Initialize.img")
    meta     = parse_idrisi_doc(DOC_PATH)
    img_vals = parse_idrisi_img_ascii(IMG_PATH)
    print("📄 Initialize.doc i Initialize.img llegits correctament.")
    print(f"   Valors únics al .img: {sorted(set(img_vals))}")
    for k, v in meta.items():
        print(f"   {k:20s}: {v}")

---
## 4. Lectura de fitxers IDRISI31 vectorials — Capa de Vegetació

Els fitxers `vegetation.dvc` i `vegetation.vec` defineixen polígons en coordenades normalitzades (0–100).  
S'utilitza **ray-casting** per rasteritzar-los sobre la graella. Si no existeixen, s'usa la capa procedural.

In [ ]:
def parse_vec(path):
    """Llegeix un fitxer .vec IDRISI31 i retorna una llista de polígons."""
    polygons = []
    with open(path, "r", errors="ignore") as f:
        lines = [l.strip() for l in f if l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) < 2:
            i += 1; continue
        try:
            poly_id = int(parts[0])
            int(parts[1])
        except ValueError:
            i += 1; continue
        i += 1
        points = []
        while i < len(lines):
            xy = lines[i].split(); i += 1
            if len(xy) < 2: continue
            x, y = float(xy[0]), float(xy[1])
            if x == 0.0 and y == 0.0: break
            points.append((x, y))
        if points:
            polygons.append({"id": poly_id, "points": points})
    return polygons


def point_in_polygon(px, py, pts):
    """Ray-casting: retorna True si (px, py) és dins del polígon."""
    n, inside, j = len(pts), False, len(pts) - 1
    for i in range(n):
        xi, yi = pts[i]; xj, yj = pts[j]
        if ((yi > py) != (yj > py)) and (px < (xj-xi)*(py-yi)/(yj-yi+1e-12)+xi):
            inside = not inside
        j = i
    return inside


def build_vegetation_grid_vec(polygons, rows, cols, min_x, max_x, min_y, max_y):
    """Rasteritza els polígons vectorials a una graella rows×cols via ray-casting."""
    grid = np.full((rows, cols), DEFAULT_VEGETATION, dtype=float)
    for r in range(rows):
        for c in range(cols):
            px = (c + 0.5) / cols * (max_x - min_x) + min_x
            py = (r + 0.5) / rows * (max_y - min_y) + min_y
            for poly in polygons:
                if point_in_polygon(px, py, poly["points"]):
                    grid[r, c] = POLYGON_VEGETATION.get(poly["id"], DEFAULT_VEGETATION)
                    break
    return grid


if MODE == "fitxers":
    VEC_PATH = os.path.join(BASE_DIR, "vegetation.vec")
    DVC_PATH = os.path.join(BASE_DIR, "vegetation.dvc")
    veg_meta = parse_idrisi_doc(DVC_PATH)
    polygons = parse_vec(VEC_PATH)
    print(f"🗺️  vegetation.vec llegit: {len(polygons)} polígon(s)")
    for p in polygons:
        print(f"   id={p['id']:3d} → {POLYGON_VEGETATION.get(p['id'], DEFAULT_VEGETATION):2d}h combustió, "
              f"{len(p['points'])} punts")

---
## 5. Construcció de les capes

Combinem les fonts disponibles per obtenir les capes finals.  
Les **zones de llac** (humitat = 5.0) s'identifiquen com a `WATER` i mai es cremen.

In [ ]:
# ══════════════════════════════════════════════════════
#  PARÀMETRES DE LA SIMULACIÓ (modifica aquí!)
# ══════════════════════════════════════════════════════
IGNITE_CELLS = [(5, 5), (5, 15), (10, 10)]  # focus d'ignició (fila, col)
WIND_DIR     = (0, 1)   # (Δfila, Δcol): (0,1)=est, (1,0)=sud, (0,0)=sense vent
MAX_STEPS    = 80       # passos màxims de simulació
# ══════════════════════════════════════════════════════

if MODE == "fitxers":
    # Llegim la mida directament de la capçalera Initialize.doc
    ROWS = int(meta.get("rows", 20))
    COLS = int(meta.get("columns", 20))
    hum_grid = build_humidity_grid_ascii(img_vals, ROWS, COLS)
    min_x = float(veg_meta.get("min. x", 0))
    max_x = float(veg_meta.get("max. x", 100))
    min_y = float(veg_meta.get("min. y", 0))
    max_y = float(veg_meta.get("max. y", 100))
    veg_grid = build_vegetation_grid_vec(polygons, ROWS, COLS, min_x, max_x, min_y, max_y)
else:
    ROWS, COLS = ROWS_PROC, COLS_PROC
    hum_grid   = hum_proc.copy()
    veg_grid   = veg_proc.copy()

water_mask = (hum_grid >= 5.0)
n_llacs    = int(np.sum(water_mask))

print(f"✅ Graella {ROWS}×{COLS} preparada (mode: {MODE})")
print(f"   Humitat    min/max : {hum_grid.min():.1f} / {hum_grid.max():.1f} hores")
print(f"   Vegetació  min/max : {veg_grid.min():.1f} / {veg_grid.max():.1f} hores")
print(f"   Cel·les d'aigua    : {n_llacs} ({n_llacs/(ROWS*COLS)*100:.1f}%)")
print(f"   Focus d'ignició    : {len(IGNITE_CELLS)} → {IGNITE_CELLS}")

---
## 6. Visualització de les capes inicials

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Capes d'entrada — Estat inicial", fontsize=14, fontweight='bold')

im0 = axes[0].imshow(hum_grid, cmap="Blues", vmin=0, vmax=5)
axes[0].set_title("Capa d'Humitat (hores)")
axes[0].set_xlabel("Columna"); axes[0].set_ylabel("Fila")
plt.colorbar(im0, ax=axes[0], shrink=0.85, label="Hores")

im1 = axes[1].imshow(veg_grid, cmap="Greens", vmin=0, vmax=20)
axes[1].set_title("Capa de Vegetació (hores de combustió)")
axes[1].set_xlabel("Columna"); axes[1].set_ylabel("Fila")
plt.colorbar(im1, ax=axes[1], shrink=0.85, label="Hores")

# Capa combinada: mostra la humitat, ressalta les zones d'aigua
hum_display = hum_grid.copy()
im2 = axes[2].imshow(hum_display, cmap="YlOrBr_r", vmin=0, vmax=5)
axes[2].contour(water_mask, levels=[0.5], colors='dodgerblue', linewidths=2)
axes[2].set_title("Zones d'aigua (contorn blau) + humitat")
axes[2].set_xlabel("Columna"); axes[2].set_ylabel("Fila")
plt.colorbar(im2, ax=axes[2], shrink=0.85, label="Humitat")

# Marca els focus d'ignició
for (r, c) in IGNITE_CELLS:
    if c < COLS and r < ROWS:
        for ax in axes:
            ax.plot(c, r, 'r*', markersize=10)

plt.tight_layout()
plt.show()

---
## 7. Model m:n-CA^k — Autòmat Cel·lular

### Regles de transició

| Estat actual | Condició | Estat següent |
|---|---|---|
| `EMPTY` | Veí cremant + humitat > 0 | `MOIST` (comença a escalfar-se) |
| `EMPTY` | Veí cremant + humitat = 0 | `BURNING` (s'encén directament) |
| `MOIST` | `hum_timer` ≥ `humitat_inicial` | `BURNING` (humitat esgotada) |
| `BURNING` | `burn_timer` ≥ `vegetació_inicial` | `BURNED` (vegetació consumida) |
| `WATER` | — | `WATER` (immutable) |

La propagació té una **probabilitat base del 85%**, modulada pel vent.

In [ ]:
class WildfireCA:
    """
    Model m:n-CA^k per a la propagació d'un incendi forestal.

    Capes:
      state       : CellState de cada cel·la (inclou WATER)
      hum_init    : humitat inicial (hores)
      veg_init    : vegetació inicial (hores de combustió)
      burn_timer  : comptador de passos cremant
      hum_timer   : comptador de passos escalfant-se
    """

    def __init__(self, humidity_grid, vegetation_grid, wind_dir=(0, 0)):
        self.rows, self.cols = humidity_grid.shape
        self.hum_init  = humidity_grid.copy().astype(float)
        self.veg_init  = vegetation_grid.copy().astype(float)
        self.wind_dir  = wind_dir
        self.max_veg   = max(1.0, float(np.max(vegetation_grid)))
        self.reset()

    def reset(self):
        self.state      = np.full((self.rows, self.cols), CellState.EMPTY, dtype=int)
        self.burn_timer = np.zeros((self.rows, self.cols), dtype=float)
        self.hum_timer  = np.zeros((self.rows, self.cols), dtype=float)
        self.veg_cur    = self.veg_init.copy()   # vegetació restant (per al gradient de color)
        self.hum_cur    = self.hum_init.copy()   # humitat restant
        # Zones d'aigua → estat permanent WATER
        self.state[self.hum_init >= 5.0] = CellState.WATER
        self.step    = 0
        self.history = []

    def ignite(self, row, col):
        """Encén manualment la cel·la (row, col) si no és aigua."""
        if 0 <= row < self.rows and 0 <= col < self.cols:
            if self.state[row, col] not in (CellState.WATER, CellState.BURNED):
                self.state[row, col]      = CellState.BURNING
                self.burn_timer[row, col] = 0
                print(f"  🔥 [{row:3d},{col:3d}]  veg={self.veg_init[row,col]:.0f}h  "
                      f"hum={self.hum_init[row,col]:.1f}h")
            else:
                print(f"  ⚠️  [{row},{col}] és zona d'aigua o ja cremada — ignorat.")

    def _spread_prob(self, dr, dc):
        """Probabilitat de propagació, modificada direccionalment pel vent."""
        base = 0.85
        wr, wc = self.wind_dir
        if wr != 0 or wc != 0:
            dot  = dr * wr + dc * wc
            norm = (wr**2 + wc**2) ** 0.5
            if norm > 0:
                boost = dot / norm * 0.12
                base  = min(0.98, max(0.30, base + boost))
        return base

    def advance(self):
        """Executa un pas de l'autòmat cel·lular."""
        new_state      = self.state.copy()
        new_burn_timer = self.burn_timer.copy()
        new_hum_timer  = self.hum_timer.copy()
        new_veg_cur    = self.veg_cur.copy()
        new_hum_cur    = self.hum_cur.copy()

        for r in range(self.rows):
            for c in range(self.cols):
                s = self.state[r, c]
                if s == CellState.WATER:
                    continue   # les zones d'aigua no canvien mai

                elif s == CellState.BURNING:
                    new_burn_timer[r, c] += 1
                    new_veg_cur[r, c]     = max(0.0, self.veg_init[r, c] - new_burn_timer[r, c])
                    if new_burn_timer[r, c] >= self.veg_init[r, c]:
                        new_state[r, c] = CellState.BURNED
                    else:
                        for dr in [-1, 0, 1]:
                            for dc in [-1, 0, 1]:
                                if dr == 0 and dc == 0:
                                    continue
                                nr, nc = r + dr, c + dc
                                if not (0 <= nr < self.rows and 0 <= nc < self.cols):
                                    continue
                                if self.state[nr, nc] != CellState.EMPTY:
                                    continue
                                if np.random.random() < self._spread_prob(dr, dc):
                                    if self.hum_init[nr, nc] > 0:
                                        new_state[nr, nc]     = CellState.MOIST
                                        new_hum_timer[nr, nc] = 0
                                    else:
                                        new_state[nr, nc]      = CellState.BURNING
                                        new_burn_timer[nr, nc] = 0

                elif s == CellState.MOIST:
                    new_hum_timer[r, c] += 1
                    new_hum_cur[r, c]    = max(0.0, self.hum_init[r, c] - new_hum_timer[r, c])
                    if new_hum_timer[r, c] >= self.hum_init[r, c]:
                        new_state[r, c]      = CellState.BURNING
                        new_burn_timer[r, c] = 0

        self.state      = new_state
        self.burn_timer = new_burn_timer
        self.hum_timer  = new_hum_timer
        self.veg_cur    = new_veg_cur
        self.hum_cur    = new_hum_cur
        self.step      += 1
        self.history.append(self.state.copy())

    def render_rgb(self):
        """
        Retorna una imatge RGB (H×W×3) amb gradient de color continu:
        - Verd intens → verd clar: vegetació sana (alta → baixa)
        - Rosa → taronja → vermell: escalfant-se → cremant
        - Gris fosc: cremat
        - Blau: massa d'aigua
        """
        img = np.zeros((self.rows, self.cols, 3), dtype=np.uint8)
        for r in range(self.rows):
            for c in range(self.cols):
                s = self.state[r, c]
                if s == CellState.WATER:
                    img[r, c] = [64, 164, 223]
                elif s == CellState.EMPTY:
                    ratio = min(1.0, self.veg_cur[r, c] / self.max_veg)
                    g     = int(80 + ratio * 150)
                    img[r, c] = [0, g, 0]
                elif s == CellState.MOIST:
                    ratio = 1.0 - min(1.0, self.hum_timer[r, c] /
                                      max(1, self.hum_init[r, c]))
                    img[r, c] = [int(220 * (1 - ratio)), int(100 * ratio), int(180 * ratio)]
                elif s == CellState.BURNING:
                    ratio = 1.0 - min(1.0, self.veg_cur[r, c] / max(1, self.veg_init[r, c]))
                    g     = int(200 * (1 - ratio))
                    img[r, c] = [255, g, 0]
                else:  # BURNED
                    img[r, c] = [50, 50, 50]
        return img

    def stats(self):
        total   = self.rows * self.cols
        water   = int(np.sum(self.state == CellState.WATER))
        burning = int(np.sum(self.state == CellState.BURNING))
        moist   = int(np.sum(self.state == CellState.MOIST))
        burned  = int(np.sum(self.state == CellState.BURNED))
        safe    = total - burning - moist - burned - water
        return {"step": self.step, "burning": burning, "moist": moist,
                "burned": burned, "safe": safe, "water": water, "total": total}

    def is_active(self):
        return (np.any(self.state == CellState.BURNING) or
                np.any(self.state == CellState.MOIST))

print("✅ Classe WildfireCA definida (amb suport d'aigua i gradient de color).")

---
## 8. Execució de la simulació (múltiples focus)

In [ ]:
np.random.seed(42)

ca = WildfireCA(hum_grid, veg_grid, wind_dir=WIND_DIR)

print(f"🔥 Encenent {len(IGNITE_CELLS)} focus d'ignició...")
for (r, c) in IGNITE_CELLS:
    if r < ROWS and c < COLS:
        ca.ignite(r, c)

stats_history = []

print(f"\n{'Pas':>5}  {'🔥Flames':>9}  {'💧Escalfant':>11}  {'⬛Cremat':>9}  {'🌿Sa':>7}  {'%Cremat':>8}")
print("-" * 62)

for _ in range(MAX_STEPS):
    if not ca.is_active():
        print("  → Foc extingit.")
        break
    ca.advance()
    s = ca.stats()
    stats_history.append(s)
    pct = s["burned"] / s["total"] * 100
    print(f"{s['step']:>5}  {s['burning']:>9}  {s['moist']:>11}  "
          f"{s['burned']:>9}  {s['safe']:>7}  {pct:>7.1f}%")

print("-" * 62)
sf = ca.stats()
flamable = sf['total'] - sf['water']
print(f"\n📊 Resultat final — Pas {sf['step']}:")
print(f"   🔥 Cremat   : {sf['burned']:5d} cel·les ({sf['burned']/flamable*100:.1f}% del terreny inflamable)")
print(f"   🌿 Sa       : {sf['safe']:5d} cel·les ({sf['safe']/flamable*100:.1f}%)")
print(f"   💧 Aigua    : {sf['water']:5d} cel·les (protegides)")

---
## 9. Visualització del resultat final (amb gradient de color)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f"Simulació d'Incendi Forestal — Resultat Final (Pas {ca.step})",
             fontsize=14, fontweight='bold')

im0 = axes[0].imshow(hum_grid, cmap="Blues", vmin=0, vmax=5)
axes[0].set_title("Capa d'Humitat (inicial)", fontsize=11)
axes[0].set_xlabel("Columna"); axes[0].set_ylabel("Fila")
plt.colorbar(im0, ax=axes[0], shrink=0.8, label="Hores")

im1 = axes[1].imshow(veg_grid, cmap="Greens", vmin=0, vmax=20)
axes[1].set_title("Capa de Vegetació (inicial)", fontsize=11)
axes[1].set_xlabel("Columna"); axes[1].set_ylabel("Fila")
plt.colorbar(im1, ax=axes[1], shrink=0.8, label="Hores")

# Resultat amb gradient de color continu
axes[2].imshow(ca.render_rgb())
axes[2].set_title("Propagació final (gradient de color)", fontsize=11)
axes[2].set_xlabel("Columna"); axes[2].set_ylabel("Fila")
axes[2].legend(handles=LEGEND_PATCHES, loc="upper right", fontsize=8, framealpha=0.85)
for (r, c) in IGNITE_CELLS:
    if r < ROWS and c < COLS:
        axes[2].plot(c, r, 'w*', markersize=14, markeredgecolor='black', markeredgewidth=0.5)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/resultat_final.png", dpi=130, bbox_inches="tight")
plt.show()
print("💾 Figura guardada: resultat_final.png")

---
## 10. Evolució temporal de la simulació

In [ ]:
if stats_history:
    steps    = [s["step"]   for s in stats_history]
    burning  = [s["burning"] for s in stats_history]
    moist    = [s["moist"]   for s in stats_history]
    burned   = [s["burned"]  for s in stats_history]
    safe     = [s["safe"]    for s in stats_history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("Evolució temporal de la simulació", fontsize=13, fontweight='bold')

    # Cel·les per estat
    ax1.plot(steps, burning, color="#EF9F27", lw=2,   label="En flames")
    ax1.plot(steps, moist,   color="#F4C0D1", lw=2,   label="Escalfant-se")
    ax1.plot(steps, burned,  color="#444441", lw=2,   label="Cremat")
    ax1.plot(steps, safe,    color="#639922", lw=2,   label="Sa")
    ax1.set_xlabel("Pas de temps")
    ax1.set_ylabel("Nombre de cel·les")
    ax1.set_title("Cel·les per estat")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Percentatge cremat acumulat
    total = ca.rows * ca.cols
    pct_burned = [b / total * 100 for b in burned]
    ax2.fill_between(steps, pct_burned, color="#444441", alpha=0.7)
    ax2.set_xlabel("Pas de temps")
    ax2.set_ylabel("% del territori cremat")
    ax2.set_title("Superfície cremada acumulada")
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("/mnt/user-data/outputs/evolucio_temporal.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("💾 Figura guardada: evolucio_temporal.png")

---
## 11. Fotogrames de la propagació (gradient de color)

Visualitzem l'evolució pas a pas usant el render RGB amb gradients.

In [ ]:
if ca.history:
    n_frames = len(ca.history)
    indices  = np.linspace(0, n_frames - 1, min(12, n_frames), dtype=int)

    ncols = 4
    nrows = int(np.ceil(len(indices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.5))
    axes = axes.flatten()
    fig.suptitle("Propagació de l'incendi — fotogrames clau (gradient de color)",
                 fontsize=13, fontweight='bold')

    # Reconstruïm un CA temporal per renderitzar cada fotograma
    ca_tmp = WildfireCA(hum_grid, veg_grid, wind_dir=WIND_DIR)

    for i, idx in enumerate(indices):
        ca_tmp.state = ca.history[idx]
        # Estimem veg_cur a partir del burn_timer no disponible al historial
        # → usem l'estat discret per pintar colors aproximats
        img = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
        for r in range(ROWS):
            for c2 in range(COLS):
                s = ca.history[idx][r, c2]
                if s == CellState.WATER:    img[r,c2] = [64,164,223]
                elif s == CellState.EMPTY:  img[r,c2] = [60,160,60]
                elif s == CellState.MOIST:  img[r,c2] = [244,192,209]
                elif s == CellState.BURNING:img[r,c2] = [239,159,39]
                else:                       img[r,c2] = [50,50,50]

        axes[i].imshow(img)
        axes[i].set_title(f"Pas {idx + 1}", fontsize=9)
        axes[i].axis('off')
        for (r, c2) in IGNITE_CELLS:
            if r < ROWS and c2 < COLS:
                axes[i].plot(c2, r, 'w*', markersize=8)

    for j in range(len(indices), len(axes)):
        axes[j].axis('off')

    fig.legend(handles=LEGEND_PATCHES, loc='lower center',
               ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.savefig("/mnt/user-data/outputs/fotogrames_propagacio.png",
                dpi=130, bbox_inches="tight")
    plt.show()
    print("💾 Figura guardada: fotogrames_propagacio.png")

---
## 12. Experiments: comparació de condicions

Comparem l'impacte del vent i del nombre de focus.

In [ ]:
conditions = [
    {"label": "Sense vent\n1 focus",   "wind": (0,0),  "ignite": [(50,50)]},
    {"label": "Vent Est (→)\n3 focus", "wind": (0,1),  "ignite": IGNITE_CELLS},
    {"label": "Vent Sud (↓)\n3 focus", "wind": (1,0),  "ignite": IGNITE_CELLS},
    {"label": "Vent SE (↘)\n3 focus",  "wind": (1,1),  "ignite": IGNITE_CELLS},
]

fig, axes = plt.subplots(1, len(conditions), figsize=(5.5 * len(conditions), 6))
fig.suptitle("Comparació: efecte del vent i nombre de focus",
             fontsize=13, fontweight='bold')

for ax, cond in zip(axes, conditions):
    np.random.seed(42)
    ca_exp = WildfireCA(hum_grid, veg_grid, wind_dir=cond["wind"])
    print(f"  Simulant: {cond['label'].replace(chr(10),' | ')}...", end=" ")
    for (r, c2) in cond["ignite"]:
        if r < ROWS and c2 < COLS:
            ca_exp.ignite(r, c2)
    for _ in range(MAX_STEPS):
        if not ca_exp.is_active(): break
        ca_exp.advance()
    s = ca_exp.stats()
    flamable = s['total'] - s['water']
    pct = s['burned'] / flamable * 100
    print(f"→ {pct:.1f}% cremat")

    ax.imshow(ca_exp.render_rgb())
    for (r, c2) in cond["ignite"]:
        if r < ROWS and c2 < COLS:
            ax.plot(c2, r, 'w*', markersize=10, markeredgecolor='k', markeredgewidth=0.5)
    ax.set_title(f"{cond['label']}\n{pct:.1f}% cremat ({s['step']} passos)", fontsize=10)
    ax.set_xlabel("Columna"); ax.set_ylabel("Fila")

fig.legend(handles=LEGEND_PATCHES, loc='lower center',
           ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.03))
plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/comparacio_vent.png", dpi=130, bbox_inches="tight")
plt.show()
print("💾 Figura guardada: comparacio_vent.png")

---
## 13. Conclusions

### Model implementat
S'ha implementat un **autòmat cel·lular m:n-CA^k** per simular la propagació d'un incendi forestal sobre una graella bidimensional de **100×100 cel·les**. El model utilitza **tres capes** principals:

1. **Capa d'humitat** (de `Initialize.doc/img` o generada proceduralment): retarda l'inici del foc. Les zones amb humitat ≥ 5.0 es tracten com a **masses d'aigua** que mai es cremen.
2. **Capa de vegetació** (de `vegetation.dvc/vec` o generada proceduralment amb gradient radial i soroll): determina durant quants passos crema cada cel·la.
3. **Capa de propagació**: conté l'estat dinàmic de cada cel·la (sana / escalfant-se / en flames / cremada / aigua).

### Observacions
- Les **masses d'aigua** actuen de tallafocs naturals: deturen completament la propagació.
- Amb **múltiples focus d'ignició**, el foc convergeix i pot superar barreres d'humitat que un sol focus no creua.
- El **vent** altera asimètricament la propagació: cap a la direcció del vent, la probabilitat de contagi arriba fins al 97%.
- Les zones de **vegetació densa** (10h de combustió) cremen llarg temps i proporcionen un focus persistent.
- El **gradient de color** a la visualització (verd intens → verd clar → taronja → gris) permet observar amb precisió quina vegetació resta a cada cel·la.

### Limitacions i possibles millores
- La topografia (pendent) no es considera; en incendis reals és tan rellevant com el vent.
- El vent és constant i uniforme; un camp vectorial variable seria més realista.
- La probabilitat de propagació podria dependre de la vegetació del veí receptor (més vegetació → més probable que s'encengui).

---
## 14. Ús de la Intel·ligència Artificial

Per al desenvolupament d'aquesta pràctica s'ha fet servir **Claude (Anthropic)** com a eina de suport.

L'ús de la IA ha estat útil principalment en tres àmbits:

1. **Estructuració del codi**: ajuda a organitzar el notebook en seccions clares i a definir l'arquitectura de la classe `WildfireCA`.
2. **Revisió i millora del model**: comparant la nostra implementació amb la d'un curs anterior, la IA ha identificat elements que faltaven (graella gran, múltiples focus, zones d'aigua, gradient de color) i ha proposat com incorporar-los mantenint l'originalitat.
3. **Depuració i explicació**: ha ajudat a entendre i corregir errors en la lectura de formats de fitxer IDRISI32/31 i en la lògica de ray-casting per als polígons vectorials.

En general, l'ús de la IA ha accelerat el procés de programació i ha permès incorporar elements que d'altra manera haurien requerit molt més temps de recerca. No obstant, les decisions de disseny del model (estats, regles de transició, tractament de l'aigua) han estat preses de forma independent, fent servir la IA com a eina de suport i verificació, no com a substitut del raonament propi.